## Baseline modeling

Testing baseline models:
- Logistic regression
- RandomForest
- XGBoost (TBA)

### Load all data

In [1]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import time

In [2]:
# Set paths and load full and reduced datasets

ROOT = Path.cwd().parent # Path is anchored relative to this notebook location

DATA = ROOT / "data" / "processed"

train_full = pd.read_csv(DATA / "training_fe_full.csv", index_col="respondent_id")
test_full = pd.read_csv(DATA / "test_fe_full.csv", index_col="respondent_id")

train_reduced = pd.read_csv(DATA / "training_fe_reduced.csv", index_col="respondent_id")
test_reduced = pd.read_csv(DATA / "test_fe_reduced.csv", index_col="respondent_id")

In [3]:
# Load full features + reduced features data sets

print(f"Full data frame: {train_full.shape[1]} train columns, {test_full.shape[1]} test columns")
print(f"Reduced data frame: {train_reduced.shape[1]} train columns, {test_reduced.shape[1]} test columns")

Full data frame: 75 train columns, 73 test columns
Reduced data frame: 58 train columns, 56 test columns


In [4]:
print(train_full.columns)

Index(['Unnamed: 0', 'h1n1_concern', 'h1n1_knowledge',
       'behavioral_antiviral_meds', 'behavioral_avoidance',
       'behavioral_face_mask', 'behavioral_wash_hands',
       'behavioral_large_gatherings', 'behavioral_outside_home',
       'behavioral_touch_face', 'doctor_recc_h1n1', 'doctor_recc_seasonal',
       'chronic_med_condition', 'child_under_6_months', 'health_worker',
       'opinion_h1n1_vacc_effective', 'opinion_h1n1_risk',
       'opinion_h1n1_sick_from_vacc', 'opinion_seas_vacc_effective',
       'opinion_seas_risk', 'opinion_seas_sick_from_vacc', 'age_group',
       'education', 'income_poverty', 'household_adults', 'household_children',
       'sex_Male', 'marital_status_Missing', 'marital_status_Not Married',
       'rent_or_own_Own', 'rent_or_own_Rent', 'health_insurance_1.0',
       'health_insurance_Missing', 'race_Hispanic', 'race_Other or Multiple',
       'race_White', 'employment_status_Missing',
       'employment_status_Not in Labor Force', 'employment_sta

In [5]:
train_reduced.columns

Index(['Unnamed: 0', 'chronic_med_condition', 'child_under_6_months',
       'health_worker', 'age_group', 'education', 'income_poverty',
       'household_adults', 'household_children', 'sex_Male',
       'marital_status_Missing', 'marital_status_Not Married',
       'rent_or_own_Own', 'rent_or_own_Rent', 'health_insurance_1.0',
       'health_insurance_Missing', 'race_Hispanic', 'race_Other or Multiple',
       'race_White', 'employment_status_Missing',
       'employment_status_Not in Labor Force', 'employment_status_Unemployed',
       'census_msa_MSA, Principle City', 'census_msa_Non-MSA',
       'hhs_geo_region_bhuqouqj', 'hhs_geo_region_dqpwygqj',
       'hhs_geo_region_fpwskwrf', 'hhs_geo_region_kbazzjca',
       'hhs_geo_region_lrircsnp', 'hhs_geo_region_lzgpxyit',
       'hhs_geo_region_mlyzmhmf', 'hhs_geo_region_oxchjgsf',
       'hhs_geo_region_qufhixun', 'employment_industry_te_h1n1_vaccine',
       'employment_industry_te_seasonal_vaccine',
       'employment_occupation_t

In [6]:
# Define features and target variables

X_full = train_full[train_full.columns[1:-2]] # Select all features starting from 1st column, except last 2
y_h1n1 = train_full["h1n1_vaccine"]
y_seasonal = train_full["seasonal_vaccine"] 

X_reduced = train_reduced[train_reduced.columns[1:-2]]
y_h1n1_r = train_reduced["h1n1_vaccine"]
y_seasonal_r = train_reduced["seasonal_vaccine"] 

In [7]:
# Perform splits for full models

from sklearn.model_selection import train_test_split

# H1N1 target 
Xh_train, Xh_val, yh_train, yh_val = train_test_split(X_full, y_h1n1, train_size = 0.8, random_state=0)

# Seasonal target
Xs_train, Xs_val, ys_train, ys_val = train_test_split(X_full, y_seasonal, train_size = 0.8, random_state=0)

In [8]:
# Perform splits for reduced models

Xhr_train, Xhr_val, yhr_train, yhr_val = train_test_split(X_reduced, y_h1n1_r, train_size = 0.8, random_state=0)

Xsr_train, Xsr_val, ysr_train, ysr_val = train_test_split(X_reduced, y_seasonal_r, train_size = 0.8, random_state=0)

### Testing full vs reduced datasets with simple logistic regression model

In [9]:
# Import test models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [10]:
# Import local trainin & eval functions

import sys, os
repo_root = os.path.abspath("..")
if repo_root not in sys.path:
    sys.path.append(repo_root)

from src.modeling import fit_model, evaluate_model, run_tests 

In [11]:
# Define test models
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=0),
    "RandomForest": RandomForestClassifier(random_state=0)
    }

# Dict of my splits
datasets = {
    "H1N1_Full": (Xh_train, Xh_val, yh_train, yh_val),
    "Seasonal_Full": (Xs_train, Xs_val, ys_train, ys_val),
    "H1N1_Reduced": (Xhr_train, Xhr_val, yhr_train, yhr_val),
    "Seasonal_Reduced": (Xsr_train, Xsr_val, ysr_train, ysr_val),
}

In [12]:
first_test = run_tests(models, datasets, verbose=True)
print(first_test)


Starting training for 8 total combinations...



Datasets:   0%|          | 0/4 [00:00<?, ?it/s]


 Dataset: H1N1_Full


 Training LogisticRegression on H1N1_Full...


✅ Finished LogisticRegression on H1N1_Full in 1.37s
 Training RandomForest on H1N1_Full...


...

✅ Finished RandomForest on H1N1_Reduced in 4.66s

 Dataset: Seasonal_Reduced


 Training LogisticRegression on Seasonal_Reduced...


✅ Finished LogisticRegression on Seasonal_Reduced in 0.83s
 Training RandomForest on Seasonal_Reduced...


Datasets: 100%|██████████| 4/4 [00:25<00:00,  6.28s/it]

✅ Finished RandomForest on Seasonal_Reduced in 4.84s

 Test of all models completed 

   accuracy  precision    recall        f1   roc_auc           dataset  \
0  0.838263   0.693671  0.468376  0.559184  0.853488         H1N1_Full   
1  0.846125   0.746459  0.450427  0.561834  0.851751         H1N1_Full   
2  0.780232   0.772879  0.747385  0.759918  0.856229     Seasonal_Full   
3  0.780232   0.774018  0.745374  0.759426  0.857420     Seasonal_Full   
4  0.818233   0.656693  0.356410  0.462050  0.811929      H1N1_Reduced   
5  0.818607   0.680431  0.323932  0.438911  0.809921      H1N1_Reduced   
6  0.766567   0.756098  0.735720  0.745770  0.845812  Seasonal_Reduced   
7  0.760389   0.750415  0.726870  0.738455  0.837386  Seasonal_Reduced   

                model  train_time_sec  
0  LogisticRegression            1.37  
1        RandomForest            5.29  
2  LogisticRegression            1.75  
3        RandomForest            5.43  
4  LogisticRegression            0.87  
5      

## First test results:


| Dataset              | Model               | Accuracy | Precision | Recall | F1 Score | ROC-AUC | Train Time (s) |
| -------------------- | ------------------- | -------- | --------- | ------ | -------- | ------- | -------------- |
| **H1N1 Full**        | Logistic Regression | 0.8383   | 0.6937    | 0.4684 | 0.5592   | 0.8535  | 1.37           |
|                      | Random Forest       | 0.8461   | 0.7465    | 0.4504 | 0.5618   | 0.8518  | 5.29           |
| **Seasonal Full**    | Logistic Regression | 0.7802   | 0.7729    | 0.7474 | 0.7599   | 0.8562  | 1.75           |
|                      | Random Forest       | 0.7802   | 0.7740    | 0.7454 | 0.7594   | 0.8574  | 5.43           |
| **H1N1 Reduced**     | Logistic Regression | 0.8182   | 0.6567    | 0.3564 | 0.4621   | 0.8119  | 0.87           |
|                      | Random Forest       | 0.8186   | 0.6804    | 0.3239 | 0.4389   | 0.8099  | 4.66           |
| **Seasonal Reduced** | Logistic Regression | 0.7666   | 0.7561    | 0.7357 | 0.7458   | 0.8458  | 0.83           |
|                      | Random Forest       | 0.7604   | 0.7504    | 0.7269 | 0.7385   | 0.8374  | 4.84           |


## Summary:
For H1N1 target the full model has slightlyn higher F1 score
For seasonal target the reduced and FULL models perform quite similarly. 

More advanced modelling and training techniques such as XGBoost or CATBoost are needed to look deeper into the feature selection process, also so we can perform hyperparameter tuning.